In [1]:
# [CELL 1] STUDENT CONFIGURATION

from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Malitao"
STUDENT_ID = "TUPM-26-1135"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

BASE_DIR = Path.cwd()
DATASET_FILE = BASE_DIR / "weblog.csv"

OUTPUT_DIR = BASE_DIR

print("STUDENT CONFIGURATION")
print("---------------------")
print("Last Name:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to Process:", RECORDS_TO_PROCESS)
print("Target Status Class:", f"{TARGET_STATUS_CLASS}xx")
print("Dataset Found:", DATASET_FILE.exists())

STUDENT CONFIGURATION
---------------------
Last Name: Malitao
Student ID: TUPM-26-1135
Seed Digit: 5
ID Sum: 18
Records to Process: 1018
Target Status Class: 3xx
Dataset Found: True


In [2]:
# [CELL 2] DATASET INSPECTION AND VALIDATION

required_columns = ["IP", "Time", "URL", "Staus"]

with open(DATASET_FILE, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    headers = reader.fieldnames

    print("DATASET HEADERS:", headers)

    if not headers:
        raise ValueError("The CSV file is empty or has no header.")

    missing_columns = [
        column for column in required_columns
        if column not in headers
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    total_dataset_records = sum(1 for row in reader)

print("\nDATASET VALIDATION")
print("------------------")
print("Required Columns:", required_columns)
print("Total Dataset Records:", total_dataset_records)
print("Missing Columns:", missing_columns)
print("Dataset Structure: VALID")

DATASET HEADERS: ['IP', 'Time', 'URL', 'Staus']

DATASET VALIDATION
------------------
Required Columns: ['IP', 'Time', 'URL', 'Staus']
Total Dataset Records: 16007
Missing Columns: []
Dataset Structure: VALID


In [3]:
# [CELL 3] READ AND FILTER WEB SERVER LOGS

selected_records = []
invalid_records = 0
processed_records = 0

with open(DATASET_FILE, "r", newline="", encoding="utf-8-sig") as file:

    reader = csv.DictReader(file)

    for row in reader:

        if processed_records >= RECORDS_TO_PROCESS:
            break

        processed_records += 1

        try:
            ip = row["IP"].strip()
            timestamp = row["Time"].strip()
            url = row["URL"].strip()
            status = row["Staus"].strip()

            # Validate required fields
            if not ip or not timestamp or not url or not status:
                invalid_records += 1
                continue

            # Validate HTTP status code
            status_code = int(status)

            if status_code < 100 or status_code > 599:
                invalid_records += 1
                continue

            # Select records belonging to the target status class
            if status_code // 100 == TARGET_STATUS_CLASS:

                selected_records.append({
                    "IP": ip,
                    "Time": timestamp,
                    "URL": url,
                    "Staus": status_code
                })

        except (ValueError, KeyError, AttributeError, TypeError):
            invalid_records += 1

print("WEB SERVER LOG PROCESSING")
print("-------------------------")
print("Records Processed:", processed_records)
print("Target Status Class:", f"{TARGET_STATUS_CLASS}xx")
print("Records in Target Class:", len(selected_records))
print("Invalid Records:", invalid_records)

WEB SERVER LOG PROCESSING
-------------------------
Records Processed: 1018
Target Status Class: 3xx
Records in Target Class: 164
Invalid Records: 0


In [4]:
# [CELL 4] STATUS CODE AND URL FREQUENCY ANALYSIS

status_counts = Counter()
url_counts = Counter()

for record in selected_records:

    status_counts[record["Staus"]] += 1
    url_counts[record["URL"]] += 1

# Get the five most frequently requested URLs
top_five_urls = url_counts.most_common(5)

print("HTTP STATUS CODE FREQUENCY")
print("--------------------------")

for status_code, count in sorted(status_counts.items()):
    print(f"HTTP {status_code}: {count} requests")

print("\nTOP 5 MOST REQUESTED URLS")
print("-------------------------")

for rank, (url, count) in enumerate(top_five_urls, start=1):
    print(f"{rank}. {url} - {count} requests")

print("\nTOTAL REQUESTS:", sum(status_counts.values()))

HTTP STATUS CODE FREQUENCY
--------------------------
HTTP 302: 143 requests
HTTP 304: 21 requests

TOP 5 MOST REQUESTED URLS
-------------------------
1. GET / HTTP/1.1 - 45 requests
2. POST /process.php HTTP/1.1 - 35 requests
3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 - 34 requests
4. GET /home.php HTTP/1.1 - 9 requests
5. GET /logout.php HTTP/1.1 - 7 requests

TOTAL REQUESTS: 164


In [5]:
# [CELL 5] EXPORT FILTERED RECORDS AND JSON SUMMARY

# Define output file paths
FILTERED_CSV = OUTPUT_DIR / "filtered_logs.csv"
SUMMARY_JSON = OUTPUT_DIR / "log_summary.json"
BACKUP_JSON = OUTPUT_DIR / "log_summary_backup.json"

# Export selected records to CSV
with open(FILTERED_CSV, "w", newline="", encoding="utf-8") as file:

    fieldnames = ["IP", "Time", "URL", "Staus"]

    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(selected_records)


# Prepare JSON summary
summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": processed_records,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": len(selected_records),
    "invalid_records": invalid_records,
    "count_per_status_code": {
        str(code): count
        for code, count in sorted(status_counts.items())
    },
    "top_five_requested_urls": [
        {
            "url": url,
            "requests": count
        }
        for url, count in top_five_urls
    ]
}

# Export JSON summary
with open(SUMMARY_JSON, "w", encoding="utf-8") as file:

    json.dump(summary, file, indent=4)


print("FILE EXPORT RESULTS")
print("-------------------")
print("Filtered CSV:", FILTERED_CSV.name)
print("JSON Summary:", SUMMARY_JSON.name)
print("CSV Exported Records:", len(selected_records))
print("JSON Summary Created: SUCCESS")

FILE EXPORT RESULTS
-------------------
Filtered CSV: filtered_logs.csv
JSON Summary: log_summary.json
CSV Exported Records: 164
JSON Summary Created: SUCCESS


In [6]:
# [CELL 6] BACKUP AND FILE VERIFICATION

# Create backup of JSON summary
shutil.copy2(SUMMARY_JSON, BACKUP_JSON)

# Verify that all files exist
print("FILE VERIFICATION")
print("-----------------")

files_to_verify = [
    FILTERED_CSV,
    SUMMARY_JSON,
    BACKUP_JSON
]

for file_path in files_to_verify:

    if file_path.exists():
        print(file_path.name, "- VERIFIED")
    else:
        print(file_path.name, "- NOT FOUND")

# Verify CSV record count
with open(FILTERED_CSV, "r", newline="", encoding="utf-8") as file:

    exported_reader = csv.DictReader(file)
    exported_count = sum(1 for row in exported_reader)

print("\nRECORD COUNT VERIFICATION")
print("------------------------")
print("Selected Records:", len(selected_records))
print("Exported Records:", exported_count)
print("Count Matches:", exported_count == len(selected_records))

# Verify backup content
with open(SUMMARY_JSON, "r", encoding="utf-8") as original:
    original_data = json.load(original)

with open(BACKUP_JSON, "r", encoding="utf-8") as backup:
    backup_data = json.load(backup)

print("\nBACKUP VERIFICATION")
print("-------------------")
print("Backup Exists:", BACKUP_JSON.exists())
print("Backup Matches Original:", original_data == backup_data)

FILE VERIFICATION
-----------------
filtered_logs.csv - VERIFIED
log_summary.json - VERIFIED
log_summary_backup.json - VERIFIED

RECORD COUNT VERIFICATION
------------------------
Selected Records: 164
Exported Records: 164
Count Matches: True

BACKUP VERIFICATION
-------------------
Backup Exists: True
Backup Matches Original: True
